# Compile ph and summary data from infotaxis runs with restricted beam movements

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
path_main = Path("../../simulations")

# Path to save output csv
path_csv = path_main / "20251119_summary_refined"
if not path_csv.exists():
    path_csv.mkdir(parents=True, exist_ok=True)

In [3]:
def load_summary(path_data, search_type, cr, br, pm_agent, pm_truth):
    return pd.read_csv(
        path_data / f"{search_type}_cr{cr}_br{br}_pmAgent{pm_agent:.0e}_pmTruth{pm_truth:.0e}_summary.csv",
        index_col=0
    )

In [4]:
def load_ph(path_data, search_type, cr, br, pm_agent, pm_truth):
    return xr.open_dataset(
        path_data / f"{search_type}_cr{cr}_br{br}_pmAgent{pm_agent:.0e}_pmTruth{pm_truth:.0e}_ph.nc"
    )

In [5]:
def get_ping_cross_p_max_th(ds, p_max_th=0.95):
    ping_cross_all = []
    for run in ds["run"].values:
        ping_cross_th = ds["p_max"].sel(run=run).dropna(dim="ping").values > p_max_th
        if ping_cross_th.sum() > 0:
            ping_cross = np.argwhere(ping_cross_th).min()
        else:
            ping_cross = 999
        ping_cross_all.append(int(ping_cross))
    return np.array(ping_cross_all)

In [6]:
def get_ping_p_max_diff_th(ds, pmax_diff_threshold=1e-5):
    criteria = {
        "pmax_repeat_N": 3,
        "max_ping_num": 500,
        "pmax_diff_threshold": pmax_diff_threshold,
    }    
    p_all = []
    for run in ds["run"].values:
        p_max = ds["p_max"].sel(run=run).dropna(dim="ping")
        for p in range(len(p_max)):
            if p - criteria["pmax_repeat_N"] < 0:
                continue
            p_max_diff = np.diff(p_max[p-criteria["pmax_repeat_N"]+1:p+1])
            if len(p_max_diff) > 1 and np.all(abs(p_max_diff) < criteria["pmax_diff_threshold"]):
                p_all.append(p)
                break
    return np.array(p_all)

In [7]:
def refine_num_pings(cr, br, pm_agent, pm_truth_all, p_max_th=0.95, pmax_diff_threshold=1e-5):
    for idx, pm in enumerate(pm_truth_all):
        print("------------------------------------------------")
        print(f"cr={cr}, br={br}, pm_agent={pm_agent}, pm_truth={pm}")

        # Load summary and ph details
        df_y = load_summary(
            path_data=path_main / "20251119_summary",
            search_type="infotaxis", cr=cr, br=br, pm_agent=pm_agent, pm_truth=pm)
        # Get num_pings when p_max crosses threshold
        ds_y = load_ph(
            path_data=path_main / "20251119_summary",
            search_type="infotaxis", cr=cr, br=br, pm_agent=pm_agent, pm_truth=pm)
        y1 = get_ping_cross_p_max_th(ds_y, p_max_th=p_max_th)
        y2 = get_ping_p_max_diff_th(ds_y, pmax_diff_threshold=pmax_diff_threshold)

        # Substitute num_pings in the runs that did not produce pmax > threshold
        # with pmax convergence num_pings
        idx_no_cross_y = y1==999
        y = y1.copy()
        y[idx_no_cross_y] = y2[idx_no_cross_y]

        # Store refined number of pings into the summary df
        df_y["num_pings_cross_pmax_th"] = y1
        df_y["num_pings_pmax_diff"] = y2
        df_y["num_pings_2conditions"] = y

        # Check if max dk cube is the same as target location at these pings
        ds_y["y1"] = (["run"], y1)
        ds_y["y2"] = (["run"], y2)
        ds_y["y"] = (["run"], y)
        df_y["success_pmax_th"] = (ds_y["dk_max_cube"].sel(ping=ds_y["y1"]) == ds_y["target_cube"]).sum(dim="cube_dim") == 3
        df_y["success_pmax_diff"] = (ds_y["dk_max_cube"].sel(ping=ds_y["y2"]) == ds_y["target_cube"]).sum(dim="cube_dim") == 3
        df_y["success_2conditions"] = (ds_y["dk_max_cube"].sel(ping=ds_y["y"]) == ds_y["target_cube"]).sum(dim="cube_dim") == 3

        fname_postfix = (
            f"cr{cr}_br{br}_pmAgent{pm_agent:.0e}_pfaAgent{pm_agent:.0e}_pmTruth{pm:.0e}_pfaTruth{pm:.0e}"
            f"_summary_pMaxTh{p_max_th:.1e}_pMaxDiffTh{pmax_diff_threshold:.0e}.csv"
        )
        df_y.to_csv(path_csv / f"infotaxis_{fname_postfix}")

        print("save refined info to:")
        print(f" - infotaxis_{fname_postfix}")

In [8]:
cr_all = [5, 10]
br_all = [1, 2]
pm_agent_all = [0.001]
pm_truth_all = [0.005, 0.01, 0.02, 0.03, 0.04, 0.05]

In [9]:
for cr in cr_all:
    for br in br_all:
        for pm_agent in pm_agent_all:
             refine_num_pings(cr, br, pm_agent, pm_truth_all)

------------------------------------------------
cr=5, br=1, pm_agent=0.001, pm_truth=0.005
save refined info to:
 - infotaxis_cr5_br1_pmAgent1e-03_pfaAgent1e-03_pmTruth5e-03_pfaTruth5e-03_summary_pMaxTh9.5e-01_pMaxDiffTh1e-05.csv
------------------------------------------------
cr=5, br=1, pm_agent=0.001, pm_truth=0.01
save refined info to:
 - infotaxis_cr5_br1_pmAgent1e-03_pfaAgent1e-03_pmTruth1e-02_pfaTruth1e-02_summary_pMaxTh9.5e-01_pMaxDiffTh1e-05.csv
------------------------------------------------
cr=5, br=1, pm_agent=0.001, pm_truth=0.02
save refined info to:
 - infotaxis_cr5_br1_pmAgent1e-03_pfaAgent1e-03_pmTruth2e-02_pfaTruth2e-02_summary_pMaxTh9.5e-01_pMaxDiffTh1e-05.csv
------------------------------------------------
cr=5, br=1, pm_agent=0.001, pm_truth=0.03
save refined info to:
 - infotaxis_cr5_br1_pmAgent1e-03_pfaAgent1e-03_pmTruth3e-02_pfaTruth3e-02_summary_pMaxTh9.5e-01_pMaxDiffTh1e-05.csv
------------------------------------------------
cr=5, br=1, pm_agent=0.001, pm